In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle
from ucimlrepo import fetch_ucirepo 
from sklearn.metrics import roc_auc_score

In [15]:
# fetch dataset 
student_performance = fetch_ucirepo(id=320) 
# this actually loads the dataset related to scores on Portuguese language courses. 

# for the mathematics course (which has 395 instead of 649 instances), you can use the following: 
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student-mat.csv"
# df = pd.read_csv(url, sep=";")
  
# data (as pandas dataframes) 
X = student_performance.data.features 
y = student_performance.data.targets 
  
df_student = pd.concat([X,y],axis=1)

In [16]:
df_student

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,MS,F,19,R,GT3,T,2,3,services,other,...,5,4,2,1,2,5,4,10,11,10
645,MS,F,18,U,LE3,T,3,1,teacher,services,...,4,3,4,1,1,1,4,15,15,16
646,MS,F,18,U,GT3,T,1,1,other,other,...,1,1,1,1,1,5,6,11,12,9
647,MS,M,17,U,LE3,T,3,1,services,services,...,2,4,5,3,4,2,6,10,10,10


In [17]:
df_student.drop(columns=["G1","G2","school","address"],inplace=True)
df_student.rename(columns={'G3':'target'},inplace=True)
df_student["target"]=df_student["target"].apply(lambda x: 0 if x>=10 else 1) #we predict the fails (1)

numeric_features=[i for i in df_student.columns if df_student[i].dtype in [np.int64, np.int64]]
categorical_features = [col for col in df_student.columns if col not in numeric_features]

feature_order=['sex', 'age', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'traveltime',
       'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime',
       'goout', 'Dalc', 'Walc', 'health', 'absences', 'guardian_father',
       'guardian_mother', 'guardian_other', 'Fjob_at_home', 'Fjob_health',
       'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_course',
       'reason_home', 'reason_other', 'reason_reputation', 'Mjob_at_home',
       'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher','target']

In [18]:
# target distribution
print(df_student['target'].value_counts())

target
0    549
1    100
Name: count, dtype: int64


In [19]:
randomstate=123

def encode_features(df, categorical_features):
    binary_mappings = {}
    for feature in categorical_features:
        unique_values = df[feature].nunique()
        if unique_values == 2:
            # Binary encoding
            df[feature] = df[feature].astype('category')
            mapping = dict(enumerate(df[feature].cat.categories))
            binary_mappings[feature] = {v: k for k, v in mapping.items()}
            df[feature] = df[feature].map(binary_mappings[feature])
            df[feature] = df[feature].astype('int')
        else:
            # One-hot encoding
            one_hot = pd.get_dummies(df[feature], prefix=feature, dtype="int")
            df = df.drop(feature, axis=1)
            df = pd.concat([df, one_hot], axis=1)
    return df, binary_mappings

# Apply encoding
df_student , binary_mappings= encode_features(df_student, categorical_features)

#Move target to end
target_col = df_student.pop("target")
df_student.insert(len(df_student.columns), "target", target_col)

#Create and save final train/test sets with target being the last column
df_student=df_student[feature_order]
student_train,student_test = train_test_split(df_student, test_size=0.2 , random_state=randomstate)

student_train.to_parquet("student_dataset/train_cleaned.parquet")
student_test.to_parquet("student_dataset/test_cleaned.parquet")

x_train = student_train.drop("target", axis=1)
y_train= student_train["target"]

x_test=student_test.drop("target", axis=1)
y_test=student_test["target"]

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=randomstate)

model.fit(x_train, y_train)

target_pred = model.predict(x_test)
target_pred_proba = model.predict_proba(x_test)[:, 1]  
# evaluate model based on accuracy and AUC metrics
accuracy = accuracy_score(y_test, target_pred)
auc = roc_auc_score(y_test, target_pred_proba)  
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"AUC: {auc * 100:.2f}%")

Accuracy: 86.15%
AUC: 81.27%


In [21]:
with open('student_dataset/RF.pkl', 'wb') as f:
    pickle.dump(model, f)

In [22]:
x_train.columns 

Index(['sex', 'age', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'traveltime',
       'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime',
       'goout', 'Dalc', 'Walc', 'health', 'absences', 'guardian_father',
       'guardian_mother', 'guardian_other', 'Fjob_at_home', 'Fjob_health',
       'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_course',
       'reason_home', 'reason_other', 'reason_reputation', 'Mjob_at_home',
       'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher'],
      dtype='str')

In [23]:
feature_desc = [
    "The sex of the student as a binary variable (0: female, 1: male)",
    "The age of the student in years",
    "Size of the family of the student (0: greater than 3 , 1 : less than 3)",
    "Parents cohabitation status (0: living apart, 1: living together)",
    "Mother's education (0: none, 1: primary education (4th grade), 2: 5th to 9th grade, 3: secondary education or 4: higher education)",
    "Father's education (0: none, 1: primary education (4th grade), 2: 5th to 9th grade, 3: secondary education or 4: higher education)",
    "Home to school travel time (1: <15 min, 2: 15 to 30 min., 3: 30 min. to 1 hour, or 4: >1 hour)",
    "Weekly study time (1: <2 hours, 2: 2 to 5 hours, 3: 5 to 10 hours, or 4: >10 hours)",
    "Number of past class failures (from 0 to 3)",
    "Student receiving extra educational support (0: no, 1:yes)",
    "Student receiving family educational support (0: no, 1: yes)",
    "Student taking extra paid classes within the course subject (0: no, 1:yes)",
    "Stdent taking part in extra-curricular activities (0:no, 1:yes)",
    "Student has attended nursery school (0:no, 1:yes)",
    "Student wants to get higher education (0:no, 1:yes)",
    "Student has internet access at home (0:no, 1:yes)",
    "Student is in a romantic relationship (0:no, 1:yes)",
    "Quality of family relationships (numeric: from 1 - very bad to 5 - excellent)",
    "Free time after school (from 1 - very low to 5 - very high)",
    "Going out with friends (from 1 - very low to 5 - very high)",
    "Workday alcohol consumption (from 1 - very low to 5 - very high)",
    "Weekend alcohol consumption (from 1 - very low to 5 - very high)",
    "Current health status (from 1 - very bad to 5 - very good)",
    "Number of school absences (actual number of absences)",
    "One-hot variable for student's guardian -- father is guardian",
    "One-hot variable for student's guardian -- mother is guardian",
    "One-hot variable for student's guardian -- neither mother or father but someone else is guardian",
    "One-hot variable for father's job -- at home",
    "One-hot variable for father's job -- care related",
    "One-hot variable for father's job -- other",
    "One-hot variable for father's job -- civil services",
    "One-hot variable for father's job -- teacher",
    "One-hot variable for reason to choose this school -- chosen for course offer",
    "One-hot variable for reason to choose this school -- chosen due to proximity to home",
    "One-hot variable for reason to choose this school -- chosen for some other reason",
    "One-hot variable for reason to choose this school -- chosen for school reputation",
    "One-hot variable for mothers's job -- at home",
    "One-hot variable for mothers's job -- care related",
    "One-hot variable for mothers's job -- other",
    "One-hot variable for mothers's job -- civil services",
    "One-hot variable for mothers's job -- teacher",
    ]

feature_desc_df = pd.DataFrame({
    "feature_name": list(x_test.columns),
    "feature_average": x_train.mean().to_list() ,
    "feature_desc": feature_desc,
})

dataset_description="The dataset contains information about students from two Portugese high schools and in particular their family situation and other habits"
target_description="The target variable represents the final year grade, transformed into whether the student passed (1) or not (0) at the end of the year"
task_description="Predict whether a student will pass"

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('student_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [24]:
feature_desc_df 

,feature_name,feature_average,feature_desc
0,sex,0.414258,The sex of the student as a binary variable (0...
1,age,16.759152,The age of the student in years
2,famsize,0.298651,Size of the family of the student (0: greater ...
3,Pstatus,0.886320,"Parents cohabitation status (0: living apart, ..."
4,Medu,2.533719,"Mother's education (0: none, 1: primary educat..."
5,Fedu,2.350674,"Father's education (0: none, 1: primary educat..."
6,traveltime,1.568401,"Home to school travel time (1: <15 min, 2: 15 ..."
7,studytime,1.955684,"Weekly study time (1: <2 hours, 2: 2 to 5 hour..."
8,failures,0.217726,Number of past class failures (from 0 to 3)
9,schoolsup,0.113680,Student receiving extra educational support (0...
